In [1]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY    = os.getenv("GROQ_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Groq key loaded:   ", "✅" if GROQ_API_KEY    else "❌ Missing!")

python-dotenv could not parse statement starting at line 8


PageIndex key loaded: ✅
Groq key loaded:    ✅


In [2]:
from pageindex import PageIndexClient
from langchain_groq import ChatGroq

pi_client  = PageIndexClient(api_key=PAGEINDEX_API_KEY)
print("✅ PageIndex client ready")


✅ PageIndex client ready


In [3]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./OS_LAB_Manual.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ./OS_LAB_Manual.pdf
✅ Uploaded!
📋 Document ID: pi-cmppjmq6l00yn01p9p8seg49n
   (Save this ID — you'll use it throughout the notebook)


In [4]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


In [5]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "OPERATING SYSTEMS LABORATORY",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "This document is a practical file for the Operating Systems Laboratory (ITL-251) at the National Institute of Technology, Srinagar, submitted by student Harshit Kamriya. It includes administrative submission details and a comprehensive table of contents outlining nine laboratory experiments ranging from basic Linux environment setup and command execution to advanced topics like process synchronization, CPU scheduling, memory management, and disk scheduling.",
  "text": "# OPERATING SYSTEMS LABORATORY\n\nLab Manual / Practical File\n\n**Course Code:** ITL-251\n\nB.Tech \u2014 4th Semester\n\n|  Submitted By | Harshit Kamriya  |\n| --- | --- |\n|  Enrollment Number | 2024BITE037  |\n|  Department | Information Technology  |\n|  Section / Batch | IT-A  |\n|  Batch (Year) | 2024 \u2013 2028  |\n|  Semester | 4th Semester  |\n|  Academic

In [6]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] OPERATING SYSTEMS LABORATORY  (p.1)
  └─ [0001] Setting Up the Linux Environment (WSL Ubuntu)  (p.3)
  └─ [0002] Study and Execution of Basic UNIX / Linux Commands  (p.4)
  └─ [0003] Process Creation Using fork() and File I/O System Calls  (p.10)
  └─ [0004] Multithreading with POSIX Threads and Semaphores  (p.13)
  └─ [0005] Process Synchronization — Mutex Locks and Peterson's Algorithm  (p.17)
  └─ [0006] CPU Scheduling Algorithms (with and without Arrival Time)  (p.20)
    └─ [0007] Main Objective  (p.20)
    └─ [0008] Question 1  (p.20)
    └─ [0009] Question 2  (p.21)
    └─ [0010] Question 3  (p.22)
    └─ [0011] Question 4  (p.24)
    └─ [0012] Question 5  (p.25)
    └─ [0013] Question 6  (p.27)
    └─ [0014] Question 7  (p.29)
    └─ [0015] Question 8  (p.31)
  └─ [0016] Memory Management — Partitioning and Page Replacement Algorithms  (p.33)
    └─ [0017] Main Objective  (p.33)
    └─ [0018] Question 1  (p.33)
    └─ [0019] Question 2  (p.35)

In [7]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 31
   Each node = one retrievable section of the document


In [8]:
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY)

In [9]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "qwen/qwen3-32b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [10]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is race condition"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is race condition

🧠 LLM Reasoning:
The query asks about 'race condition' which is a synchronization issue in concurrent systems. The document's most relevant section is 'Process Synchronization – Mutex Locks and Peterson's Algorithm' (node 0005), as it directly addresses synchronization mechanisms that resolve race conditions. Node 0004 (multithreading with semaphores) is tangentially related but less specific. No other nodes discuss race conditions explicitly.

🎯 Selected Node IDs: ['0005']


In [11]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [12]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "qwen/qwen3-32b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [13]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [14]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="How to write first come first serve in scheduling",
    tree=pageindex_tree
)

🔍 Query: How to write first come first serve in scheduling

🧠 Reasoning: The query is about implementing First-Come-First-Serve (FCFS) scheduling. FCFS is a scheduling algorithm that processes tasks in the order they arrive. In the document tree, node 0006 (CPU Scheduling ...
🎯 Retrieved node IDs: ['0008', '0025']
📄 Sections found: ['Question 1', 'Question 1']

📝 Answer:
<think>
Okay, the user is asking how to write "first come first serve" in scheduling. Let me look at the provided context.

In the context, there are two sections under Question 1, both dealing with FCFS. The first section (Page 20) is about CPU scheduling where processes are handled in the order they arrive, all starting at time 0. The code provided calculates waiting and turnaround times. The second section (Page 42) discusses disk scheduling, where the disk head services requests in the order they arrive. Both implementations follow the FCFS principle by processing items sequentially as they come in.

For the answer

In [16]:
answer = vectorless_rag(
    query="How to write shortest job first  in scheduling",
    tree=pageindex_tree
)

🔍 Query: How to write shortest job first  in scheduling

🧠 Reasoning: The query is about implementing the Shortest Job First (SJF) scheduling algorithm. The document's node 0006 (CPU Scheduling Algorithms) contains specific questions on SJF. Question 2 (node 0009) direc...
🎯 Retrieved node IDs: ['0009', '0013']
📄 Sections found: ['Question 2', 'Question 6']

📝 Answer:
<think>
Okay, let me try to figure out how to answer the question about writing the Shortest Job First (SJF) scheduling algorithm based on the provided context.

First, the user is asking how to write SJF in scheduling. The context has two sections: Question 2 and Question 6. Question 2 deals with SJF without arrival times (all processes arrive at time 0), and Question 6 includes arrival times. 

Looking at Question 2 (Page 21), the code uses merge sort to sort processes by burst time. The main steps are: input the number of processes and their burst times, sort them using merge sort, then calculate waiting and turnaround